# Chapter 6: Advanced Logistic Regression and Extensions

## Ringkasan Bab & Penjelasan Teoretis
Regresi logistik memprediksi probabilitas suatu observasi termasuk ke dalam kelas tertentu menggunakan transformasi fungsi sigmoid/logistik terhadap kombinasi linear fitur:
$$P(Y = 1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \dots + \beta_n X_n)}}$$

### 1. Klasifikasi Biner, Multikelas (OvR vs Multinomial), dan Regularisasi
* **One-vs-Rest (OvR)**: Melatih pengklasifikasi biner terpisah untuk setiap kelas.
* **Multinomial (SoftMax)**: Mengestimasi probabilitas seluruh kelas secara simultan menggunakan fungsi SoftMax.
* **Regularisasi ($C$)**: Parameter $C$ adalah kebalikan dari kekuatan regularisasi; nilai $C$ yang kecil memberikan penalti regularisasi ($L_1$ atau $L_2$) yang kuat.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_iris, make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, roc_curve, auc
)

# 1. Binary Logistic Regression (Breast Cancer Dataset)
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)
model = LogisticRegression(max_iter=10000).fit(X_train, y_train)
y_pred = model.predict(X_test)

class_names = ['Malignant', 'Benign']
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix (Accuracy: {accuracy_score(y_test, y_pred):.2f})')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# 2. Multiclass Logistic Regression (Iris Dataset: OvR vs Multinomial)
iris = load_iris()
X_ir_train, X_ir_test, y_ir_train, y_ir_test = train_test_split(iris.data, iris.target, test_size=0.3, random_state=2024)

ovr_model = LogisticRegression(solver='liblinear').fit(X_ir_train, y_ir_train)
multinomial_model = LogisticRegression(solver='lbfgs', max_iter=1000).fit(X_ir_train, y_ir_train)

print("OvR Accuracy:", accuracy_score(y_ir_test, ovr_model.predict(X_ir_test)))
print("Multinomial Accuracy:", accuracy_score(y_ir_test, multinomial_model.predict(X_ir_test)))

# 3. Regularisasi Ridge (L2) dan Lasso (L1) pada Logistic Regression
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X, y, test_size=0.3, random_state=2024)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_s)
X_te_s = scaler.transform(X_te_s)

ridge_log = LogisticRegression(penalty='l2', solver='liblinear', C=0.1).fit(X_tr_s, y_tr_s)
lasso_log = LogisticRegression(penalty='l1', solver='liblinear', C=0.1).fit(X_tr_s, y_tr_s)
print(f"Ridge (L2) Accuracy: {accuracy_score(y_te_s, ridge_log.predict(X_te_s)):.3f}")
print(f"Lasso (L1) Accuracy: {accuracy_score(y_te_s, lasso_log.predict(X_te_s)):.3f}")

### 2. Klasifikasi Multilabel, Co-occurrence Matrix, dan Evaluasi ROC-AUC
* **Multilabel Classification**: Setiap sampel dapat memiliki beberapa label sekaligus. Matriks *co-occurrence* memvisualisasikan frekuensi dan persentase kemunculan bersama antar-label.
* **ROC Curve & AUC**: Memplot *True Positive Rate* terhadap *False Positive Rate* di berbagai ambang batas probabilitas.

In [ ]:
# 4. Multilabel Classification
X_ml, y_ml = make_multilabel_classification(
    n_samples=1000, n_features=20, n_classes=5, n_labels=2, random_state=2024
)
X_ml_train, X_ml_test, y_ml_train, y_ml_test = train_test_split(X_ml, y_ml, test_size=0.3, random_state=2024)

y_pred_all = []
for i in range(y_ml_train.shape[1]):
    clf = LogisticRegression(solver='liblinear').fit(X_ml_train, y_ml_train[:, i])
    y_pred_all.append(clf.predict(X_ml_test))

y_ml_pred = np.array(y_pred_all).T
display(pd.DataFrame(classification_report(y_ml_test, y_ml_pred, zero_division=1, output_dict=True)).transpose())

# Co-occurrence Matrix Visualization
cooccurrence = y_ml_test.T @ y_ml_test
diag = np.diag(cooccurrence)
relative_cooccurrence = cooccurrence / diag[:, None]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cooccurrence, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title('Raw Co-occurrence Counts')
sns.heatmap(relative_cooccurrence, annot=True, fmt='.2f', cmap='RdYlBu_r', vmin=0, vmax=1, ax=ax2)
ax2.set_title('Relative Co-occurrence (Normalized)')
plt.tight_layout()
plt.show()

# 5. Evaluasi Kurva ROC dan AUC
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=2024)
eval_model = LogisticRegression(solver='liblinear').fit(X_tr, y_tr)
y_prob = eval_model.predict_proba(X_te)[:, 1]
fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()